# QVerse — Introduction to Quantum Computing & Programming
        ## Week 8: Measurement, Shots, and Statistical Thinking

        **Level:** Beginner  
        **Recommended study time:** 2–4 hours  
        **Prerequisites:** Weeks 1–7

        ### Learning objectives
        - Interpret finite-shot counts as samples.
- Compare statevector probabilities with empirical frequencies.
- Measure in different bases.
- Observe approximate 1/sqrt(N) sampling-error scaling.

        ---
        **How to use this notebook**

        1. Read the short theory sections.
        2. Make a prediction before running each guided experiment.
        3. Run and modify the code.
        4. Complete every **TODO** exercise.
        5. Finish the reflection section in your own words.

        The goal is not to memorize syntax. The goal is to connect **quantum idea → circuit → result → explanation**.

In [ ]:
# Run this only if your environment does not have the required packages.
# In a terminal, the preferred setup is:
# python -m pip install "qiskit[visualization]>=2.5" matplotlib numpy

# In a fresh Colab notebook you can instead uncomment:
# %pip install "qiskit[visualization]>=2.5" matplotlib numpy -q

## 1. Probabilities versus samples

For `|+>`, ideal theory says $P(0)=P(1)=1/2$. A finite experiment produces counts such as 493/507. That is not an error in the algorithm; it is ordinary sampling fluctuation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

def sample_plus(shots, seed=None):
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.measure_all()
    counts = StatevectorSampler(seed=seed).run([qc], shots=shots).result()[0].data.meas.get_counts()
    return counts.get("0", 0) / shots

for shots in [10, 100, 1_000, 10_000]:
    print(shots, sample_plus(shots, seed=shots))

## 2. Plot one estimate at different shot counts

In [ ]:
shots_list = [10, 30, 100, 300, 1000, 3000, 10000]
estimates = [sample_plus(n, seed=100+n) for n in shots_list]

plt.semilogx(shots_list, estimates, marker="o")
plt.axhline(0.5, linestyle="--")
plt.xlabel("Shots")
plt.ylabel("Estimated P(0)")
plt.title("Finite-shot estimate of P(0) for |+>")
plt.show()

## 3. Measurement basis

In [ ]:
# Z-basis measurement of |+>
z_measure = QuantumCircuit(1)
z_measure.h(0)
z_measure.measure_all()

# X-basis measurement of |+>: rotate X basis to Z basis using H
x_measure = QuantumCircuit(1)
x_measure.h(0)   # prepare |+>
x_measure.h(0)   # basis rotation before standard measurement
x_measure.measure_all()

sampler = StatevectorSampler(seed=9)
z_counts = sampler.run([z_measure], shots=500).result()[0].data.meas.get_counts()
x_counts = sampler.run([x_measure], shots=500).result()[0].data.meas.get_counts()

print("Z basis:", z_counts)
print("X basis:", x_counts)

## 4. Empirical error scaling

For a binary probability near 1/2, the standard error of the estimated probability is approximately

$$
\sigma_{\hat p}\approx \frac{1}{2\sqrt N}.
$$

We can test the trend by repeating many independent experiments.

In [ ]:
shot_values = np.array([20, 50, 100, 200, 500, 1000, 2000])
rms_errors = []

for shots in shot_values:
    estimates = [sample_plus(int(shots), seed=10000 + shots*100 + rep) for rep in range(100)]
    rms = np.sqrt(np.mean((np.array(estimates) - 0.5)**2))
    rms_errors.append(rms)

plt.loglog(shot_values, rms_errors, marker="o", label="empirical RMS error")
plt.loglog(shot_values, 0.5/np.sqrt(shot_values), linestyle="--", label="~1/(2√N)")
plt.xlabel("Shots N")
plt.ylabel("Probability-estimate error")
plt.legend()
plt.show()

## Core exercises
1. Explain why two correct executions can return different counts.
2. Run `|+>` with 10, 100, 1,000, and 10,000 shots and record the error in estimated `P(0)`.
3. Measure `|+>` in the X basis and explain why the result becomes deterministic.
4. Prepare `|->` and compare its Z-basis and X-basis measurement statistics.

In [ ]:
# TODO: Write your solutions here.
# Add extra code cells when useful.

## Optional stretch challenge
Repeat the sampling experiment for a state with `P(1)=0.1`. Compare the observed error with the Bernoulli standard-error formula `sqrt(p(1-p)/N)`.

In [ ]:
# OPTIONAL TODO: Attempt the stretch challenge here.

## Weekly reflection
- What is a shot?
- Why is 50/50 theory not the same as exactly equal counts?
- How do we implement a measurement in a basis other than Z using gates?

## Submission checklist
- [ ] I made at least one prediction before executing a circuit.
- [ ] All guided examples run.
- [ ] I completed the core exercises.
- [ ] I explained the important output rather than only displaying it.
- [ ] My notebook is readable from top to bottom.